In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from src.data_loader import load_data, basic_cleaning

sns.set_style("whitegrid")

plt.rcParams["figure.figsize"] = (12, 6)

# Business Understanding

AlphaCare Insurance Solutions (ACIS) aims to improve its pricing strategy and identify low-risk customer segments using historical insurance claim data.

The primary business objective is to develop evidence-driven insights that can support:

- Risk-based pricing
- Customer segmentation
- Marketing optimization
- Portfolio profitability improvement

Two key business metrics are used throughout the analysis:

- **Loss Ratio = TotalClaims / TotalPremium**
- **Margin = TotalPremium − TotalClaims**

These metrics help identify profitable and high-risk customer groups.

In [ ]:
DATA_PATH = "../data/insurance_data.csv"

df = load_data(DATA_PATH)

df = basic_cleaning(df)

df.head()

In [ ]:
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

In [ ]:
df.columns.tolist()

In [ ]:
df.dtypes.sort_values()

In [ ]:
missing = (
    df.isnull()
    .sum()
    .sort_values(ascending=False)
)

missing_percent = (
    (df.isnull().sum() / len(df)) * 100
).sort_values(ascending=False)

missing_df = pd.DataFrame({
    "MissingValues": missing,
    "MissingPercent": missing_percent
})

missing_df[missing_df["MissingValues"] > 0]

# Missing Value Handling Strategy

The dataset contains missing values across several categorical and numerical variables.

Handling strategy:

- Numerical variables:
  - Median imputation for skewed distributions
  - Mean imputation for near-normal distributions

- Categorical variables:
  - Fill with `"Unknown"` category

- Date variables:
  - Converted using `errors='coerce'`

- High-missing columns:
  - Will be evaluated for exclusion during modeling if missingness is excessive.

In [ ]:
numerical_cols = df.select_dtypes(include=np.number).columns

df[numerical_cols].describe().T

In [ ]:
financial_cols = [
    "TotalPremium",
    "TotalClaims",
    "Margin",
    "LossRatio"
]

df[financial_cols].describe().T

In [ ]:
overall_loss_ratio = (
    df["TotalClaims"].sum() /
    df["TotalPremium"].sum()
)

print(f"Overall Portfolio Loss Ratio: {overall_loss_ratio:.2%}")

In [ ]:
cols = [
    "TotalPremium",
    "TotalClaims",
    "CustomValueEstimate"
]

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for i, col in enumerate(cols):
    sns.histplot(df[col], kde=True, ax=axes[i])
    axes[i].set_title(f"Distribution of {col}")

plt.tight_layout()
plt.show()

In [ ]:
top_vehicle_types = (
    df["VehicleType"]
    .value_counts()
    .head(10)
)

sns.barplot(
    x=top_vehicle_types.values,
    y=top_vehicle_types.index
)

plt.title("Top Vehicle Types")
plt.xlabel("Count")
plt.ylabel("Vehicle Type")

plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

sns.boxplot(x=df["TotalClaims"], ax=axes[0])
axes[0].set_title("Boxplot of TotalClaims")

sns.boxplot(x=df["CustomValueEstimate"], ax=axes[1])
axes[1].set_title("Boxplot of CustomValueEstimate")

plt.tight_layout()
plt.show()

In [ ]:
corr_cols = [
    "TotalPremium",
    "TotalClaims",
    "Margin",
    "LossRatio",
    "CustomValueEstimate"
]

corr = df[corr_cols].corr()

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Matrix")
plt.show()

In [ ]:
sample_df = df.sample(5000, random_state=42)

sns.scatterplot(
    data=sample_df,
    x="TotalPremium",
    y="TotalClaims",
    alpha=0.6
)

plt.title("TotalPremium vs TotalClaims")
plt.show()

In [ ]:
province_loss = (
    df.groupby("Province")
    .agg({
        "TotalPremium": "sum",
        "TotalClaims": "sum"
    })
)

province_loss["LossRatio"] = (
    province_loss["TotalClaims"] /
    province_loss["TotalPremium"]
)

province_loss = province_loss.sort_values(
    "LossRatio",
    ascending=False
)

province_loss

In [ ]:
sns.barplot(
    data=province_loss.reset_index(),
    x="LossRatio",
    y="Province"
)

plt.title("Loss Ratio by Province")
plt.xlabel("Loss Ratio")
plt.ylabel("Province")

plt.show()

In [ ]:
vehicle_risk = (
    df.groupby("VehicleType")
    .agg({
        "TotalClaims": "mean",
        "LossRatio": "mean"
    })
    .sort_values("TotalClaims", ascending=False)
    .head(10)
)

vehicle_risk

In [ ]:
gender_risk = (
    df.groupby("Gender")
    .agg({
        "LossRatio": "mean",
        "TotalClaims": "mean"
    })
)

gender_risk

In [ ]:
monthly_trends = (
    df.groupby(df["TransactionMonth"].dt.to_period("M"))
    .agg({
        "TotalClaims": "sum",
        "HasClaim": "mean"
    })
)

monthly_trends.index = monthly_trends.index.astype(str)

monthly_trends.head()

In [ ]:
monthly_trends["TotalClaims"].plot(marker="o")

plt.title("Monthly Claim Severity Trend")
plt.ylabel("Total Claims")

plt.xticks(rotation=45)

plt.show()

In [ ]:
top_makes = (
    df.groupby("Make")["TotalClaims"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
)

sns.barplot(
    x=top_makes.values,
    y=top_makes.index
)

plt.title("Vehicle Makes with Highest Average Claims")

plt.show()

In [ ]:
province_summary = (
    df.groupby("Province")
    .agg({
        "TotalPremium": "sum",
        "TotalClaims": "sum",
        "PolicyID": "count"
    })
)

province_summary["LossRatio"] = (
    province_summary["TotalClaims"] /
    province_summary["TotalPremium"]
)

sns.scatterplot(
    data=province_summary,
    x="TotalPremium",
    y="TotalClaims",
    size="PolicyID",
    hue="LossRatio",
    sizes=(100, 1000)
)

plt.title("Province Profitability Risk Map")

plt.show()

# Key Insights and Recommendations

## Key Findings

- Several provinces exhibit significantly higher loss ratios, indicating elevated insurance risk exposure.
- Certain vehicle types and makes consistently produce higher average claim amounts.
- The dataset contains extreme outliers in TotalClaims and CustomValueEstimate that may influence predictive modeling.
- Temporal analysis suggests fluctuations in claim severity across months, potentially linked to seasonality or operational factors.
- Premium and claim relationships vary considerably across geographic regions.

## Business Recommendations

- Implement province-specific pricing adjustments for high-loss regions.
- Introduce targeted marketing campaigns toward low-risk customer segments.
- Review underwriting rules for high-risk vehicle categories.
- Incorporate vehicle age and regional risk into premium optimization models.
- Apply robust outlier handling before statistical modeling to improve predictive stability.